# Silver: PDFs to Markdown

Turns the raw PDFs in `bronze_pdfs` into one Markdown document per PDF, keeping titles, headings, plain text, tables, page headers/footers and page separators. It runs as the `silver_parser` task of the `ingest_financebench` job, after `bronze_pdfs`.

```
bronze_pdfs --readStream + ai_parse_document--> silver_pdfs.parsed
silver_pdfs.parsed --split into parts + ai_query--> silver_pdfs.parts --join--> silver_pdfs.markdown
```

**One table**, `silver_pdfs`:

| Column | Meaning |
| --- | --- |
| `path` | Full volume path of the PDF |
| `parsed` | `ai_parse_document` output (VARIANT): elements with type, content and page |
| `markdown` | The final Markdown document. NULL until every part is converted |
| `parts` | `MAP<INT, STRING>`: Markdown of each converted part, by part number |
| `num_parts` | How many parts the document has |

**Stages**

| Stage | What it does | Incremental by |
| --- | --- | --- |
| 1. Create the table | Defines `silver_pdfs` | `IF NOT EXISTS` |
| 2. Parse | `ai_parse_document` on each new PDF | Streaming checkpoint |
| 2b. Validate and re-parse | Finds pages the parser failed on (rate limit, internal error) and re-parses only those pages, within a page budget | Reads `error_status`; attempt cap per document |
| 3. Convert | `ai_query` per part, check the page markers, then join into `markdown` | Only missing parts are sent |

**Parameters**

| Widget | Default | Meaning |
| --- | --- | --- |
| `catalog`, `schema` | `workspace`, `finance_bench_rag` | From the bundle variables |
| `endpoint` | `databricks-qwen3-next-80b-a3b-instruct` | Model serving endpoint used by `ai_query` |
| `max_files_per_batch` | `10` | Step 2: bronze files per micro-batch |
| `pages_per_call` | `3` | Step 3: pages sent to `ai_query` per call |
| `docs_per_batch` | `50` | Step 3: documents per save point |
| `max_parts_per_run` | `1000` | Step 3: stop after this many new parts (`ai_query` budget) |
| `reparse_page_budget` | `300` | Step 2b: max pages re-parsed per run (`ai_parse_document` budget) |
| `reparse_max_attempts` | `3` | Step 2b: give up on a document after this many failed passes |
| `reparse_chunk_pages` | `20` | Step 2b: pages per parse call |
| `reparse_pause_seconds` | `10` | Step 2b: pause between parse calls |
| `reparse_doc` | empty | Step 2b: only this document (part of the file name) |
| `reparse_dry_run` | `no` | Step 2b: `yes` prints the plan and spends nothing |

In [ ]:
import json
import re
import time

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("endpoint", "databricks-qwen3-next-80b-a3b-instruct")
dbutils.widgets.text("max_files_per_batch", "10")  # step 2: bronze files per micro-batch
dbutils.widgets.text("pages_per_call", "3")        # step 3: pages sent to ai_query per call
dbutils.widgets.text("docs_per_batch", "50")       # step 3: documents per batch (save point)
dbutils.widgets.text("max_parts_per_run", "1000")  # step 3: stop after this many new parts (ai_query budget)
dbutils.widgets.text("reparse_page_budget", "300") # step 2b: max pages re-parsed per run (ai_parse_document budget)
dbutils.widgets.text("reparse_max_attempts", "3")  # step 2b: give up on a document after this many failed passes
dbutils.widgets.text("reparse_chunk_pages", "20")  # step 2b: pages per ai_parse_document call
dbutils.widgets.text("reparse_pause_seconds", "10")# step 2b: pause between calls (rate limit)
dbutils.widgets.text("reparse_doc", "")            # step 2b: only this document (part of the file name), empty = all
dbutils.widgets.dropdown("reparse_dry_run", "no", ["no", "yes"])  # step 2b: yes = only print the plan

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
endpoint = dbutils.widgets.get("endpoint")
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))
pages_per_call = int(dbutils.widgets.get("pages_per_call"))
docs_per_batch = int(dbutils.widgets.get("docs_per_batch"))
max_parts_per_run = int(dbutils.widgets.get("max_parts_per_run"))
reparse_page_budget = int(dbutils.widgets.get("reparse_page_budget"))
reparse_max_attempts = int(dbutils.widgets.get("reparse_max_attempts"))
reparse_chunk_pages = int(dbutils.widgets.get("reparse_chunk_pages"))
reparse_pause_seconds = int(dbutils.widgets.get("reparse_pause_seconds"))
reparse_doc = dbutils.widgets.get("reparse_doc")
reparse_dry_run = dbutils.widgets.get("reparse_dry_run") == "yes"

bronze = f"{catalog}.{schema}.bronze_pdfs"
silver = f"{catalog}.{schema}.silver_pdfs"
checkpoint = f"/Volumes/{catalog}/{schema}/docs/_checkpoints/silver_pdfs"

## 1. Create the table

Creates `silver_pdfs` with explicit columns, in its own cell, before anything writes to it. `IF NOT EXISTS` makes it safe to rerun.

`parse_attempts` counts how many times step 2b has re-parsed a document's failed pages, so it can give up after a few tries.

**Challenge: the table already existed with fewer columns.** The first versions only had `path`, `parsed` and `markdown`. `CREATE TABLE IF NOT EXISTS` does not add columns to an existing table.
**Solution:** the cell checks for each newer column and runs `ALTER TABLE ... ADD COLUMNS` when it is missing, so the notebook upgrades an older table in place.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver} (
  path STRING,
  parsed VARIANT,
  markdown STRING,
  parts MAP<INT, STRING>,
  num_parts INT,
  parse_attempts INT
)
""")

# Table created by an earlier version: add the columns it lacks.
for column, ddl in [("parts", "parts MAP<INT, STRING>"), ("num_parts", "num_parts INT"), ("parse_attempts", "parse_attempts INT")]:
    if column not in spark.table(silver).columns:
        spark.sql(f"ALTER TABLE {silver} ADD COLUMNS ({ddl})")

## 2. Parse with `ai_parse_document` (streaming)

Reads `bronze_pdfs` as a stream, parses each PDF with `ai_parse_document` (version 2.0) and appends the result to `silver_pdfs.parsed`. `markdown`, `parts` and `num_parts` are written as NULL and filled in step 3. The output is a VARIANT with `document.pages` and `document.elements`. Each element has a `type` (title, section_header, text, table, figure, page_header, page_footer, ...), its `content`, and a `bbox` with the page.

`ai_parse_document` has no model to choose. It runs once per whole PDF and is not split.

**Challenge: parsing is slow and billed per document, and must never be repeated.**
**Solution:** a Structured Streaming read with a checkpoint (`/Volumes/<catalog>/<schema>/docs/_checkpoints/silver_pdfs`) records which bronze rows were processed, so a rerun or redeploy only parses new PDFs. `trigger(availableNow=True)` processes everything new and stops, like a batch task.

**Challenge: one big batch loses all its work if it fails.**
**Solution:** `maxFilesPerTrigger` (`max_files_per_batch`) keeps each micro-batch small, so progress is committed often. This only helps if `bronze_pdfs` has many files. Check `numFiles` in `DESCRIBE DETAIL`.

**Rebuild:** drop `silver_pdfs` **and** delete the checkpoint folder. Dropping only the table leaves the checkpoint, so nothing would be reparsed.

In [ ]:
(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).table(bronze)
    .select(
        "path",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
        F.lit(None).cast("string").alias("markdown"),
        F.lit(None).cast("map<int,string>").alias("parts"),
        F.lit(None).cast("int").alias("num_parts"),
        F.lit(None).cast("int").alias("parse_attempts"),
    )
    .writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(silver)
    .awaitTermination()
)
print(spark.table(silver).count(), "parsed")

## 2b. Validate the parse and re-parse only the pages that failed

Step 2 parses many PDFs at once, and `ai_parse_document` answers some pages with an error instead of content. In the first full run **3,180 of 49,965 pages (6%) came back as `Request rate limit exceeded` or `Internal error`**, in 55 documents (PepsiCo 2018 lost 278 of 479 pages). Those pages have no text, so they would be missing from the Markdown. This stage finds them and parses only them again.

**Built for a limited quota (Databricks free tier).** Every step is meant to spend as little as possible:

| Guard | What it does |
| --- | --- |
| Validate first | Reads the parser's own `error_status`. A page is retried only if the error is temporary (rate limit, internal error). Permanent errors (a PDF over 500 pages, a failed conversion) are never retried |
| Re-parse only the failed pages | Builds a small PDF from just those pages (with PyMuPDF) and parses that. Re-parsing the 55 documents whole would bill 11,870 pages. This bills 3,180 |
| Page budget per run | `reparse_page_budget` (default 300 pages) caps the parse spend of one run. A big backlog is worked off over several runs |
| Small, spaced calls | `reparse_chunk_pages` pages per call (default 20), `reparse_pause_seconds` between calls, and one document at a time, so the rate limit is not hit again |
| Stop on rate limit | If more than half of a call's pages fail again, the run stops and tells you |
| Attempt cap | `parse_attempts` counts failed passes. After `reparse_max_attempts` (default 3) the document is left alone |
| Dry run | `reparse_dry_run = yes` prints the plan (which documents, how many pages) and spends nothing |
| One document | `reparse_doc` limits the run to one document, for a small test |

The fixed pages are merged back into the document's `parsed` output at the right page. The document's `markdown` is cleared, and step 3 then reconverts only the parts whose pages changed (see 3b2). Page numbering stays the same convention as before (pages with content, counted 1, 2, 3), so documents with no errors are not affected.

In [ ]:
# Documents to convert: no markdown yet, and the parser found at least one element.
# (A PDF the parser rejected has no elements, so it has nothing to send to ai_query.)
HAS_ELEMENTS = "array_size(variant_get(parsed, '$.document.elements', 'ARRAY<VARIANT>')) > 0"
CONVERTIBLE = f"markdown IS NULL AND {HAS_ELEMENTS}"

# Page errors worth another try. Anything else (page limit, failed conversion) is permanent.
RETRYABLE = "(?i)rate limit|internal error"


def retry_plan():
    """Documents with retryable page errors that still have attempts left, the most lost pages first."""
    only_doc = f"AND path LIKE '%{reparse_doc}%'" if reparse_doc else ""
    return spark.sql(f"""
    SELECT path,
           coalesce(parse_attempts, 0) AS attempts,
           count(*) AS retry_pages,
           array_size(variant_get(parsed, '$.document.pages', 'ARRAY<VARIANT>')) AS pdf_pages
    FROM {silver}, LATERAL variant_explode(parsed:error_status) err
    WHERE {HAS_ELEMENTS}
      AND err.value:error_message::STRING RLIKE '{RETRYABLE}'
      AND coalesce(parse_attempts, 0) < {reparse_max_attempts}
      {only_doc}
    GROUP BY path, coalesce(parse_attempts, 0), array_size(variant_get(parsed, '$.document.pages', 'ARRAY<VARIANT>'))
    ORDER BY retry_pages DESC
    """)


plan = retry_plan().collect()
total_retry = sum(r.retry_pages for r in plan)
print(f"{len(plan)} documents have page errors worth retrying: {total_retry} pages.")
print(f"Page budget for this run: {reparse_page_budget}. Re-parsing whole documents would cost {sum(r.pdf_pages for r in plan)} pages.")
if plan:
    display(spark.createDataFrame(plan).selectExpr("regexp_extract(path, '[^/]+$', 0) AS doc", "retry_pages", "pdf_pages", "attempts"))

### 2b-1. Re-parse some pages, and put them back

Three small functions:

- `reparse_pages` builds a PDF from the given pages (PyMuPDF), parses it, and maps the result back to the original page numbers. The parser's page ids are the 0-based PDF page indexes, and in the small PDF page *j* is the *j*-th requested page.
- `splice` adds the new elements to the document's parsed JSON, keeps reading order (by page, and within a page as parsed), renumbers the element ids, and removes the errors of the pages that were fixed. Errors that remain stay in `error_status`.
- `save_reparsed` writes the new `parsed` back, clears `markdown` so the document is converted again, and records the attempt.

In [ ]:
import fitz  # PyMuPDF


def reparse_pages(pdf_bytes, page_ids):
    """Parse only `page_ids` (0-based PDF pages) by building a small PDF from them.
    Returns (elements_by_page, failed): elements with their original page ids, and the pages that still failed."""
    src = fitz.open(stream=pdf_bytes, filetype="pdf")
    mini = fitz.open()
    for pid in page_ids:
        mini.insert_pdf(src, from_page=pid, to_page=pid)

    df = spark.createDataFrame([(bytearray(mini.tobytes()),)], "content binary")
    out = df.select(F.expr("CAST(ai_parse_document(content, map('version', '2.0')) AS STRING)")).first()[0]
    parsed = json.loads(out)

    by_page = {pid: [] for pid in page_ids}
    for e in parsed["document"]["elements"]:
        boxes = e.get("bbox") or [{"page_id": 0}]
        original = page_ids[boxes[0]["page_id"]]
        for b in e.get("bbox") or []:
            b["page_id"] = page_ids[b["page_id"]]
        by_page[original].append(e)
    failed = {page_ids[err["page_id"]] for err in (parsed.get("error_status") or []) if err["page_id"] < len(page_ids)}
    return by_page, failed


def splice(parsed_json, elements_by_page, fixed_pages):
    """Add the new elements to a document's parsed JSON and drop the errors of the fixed pages."""
    doc = json.loads(parsed_json)
    elements = doc["document"]["elements"] + [e for pid in sorted(elements_by_page) for e in elements_by_page[pid]]
    elements.sort(key=lambda e: (e.get("bbox") or [{"page_id": 0}])[0]["page_id"])  # stable: keeps order within a page
    for i, e in enumerate(elements):
        e["id"] = i
    doc["document"]["elements"] = elements
    left = [er for er in (doc.get("error_status") or []) if er["page_id"] not in fixed_pages]
    doc["error_status"] = left or None
    return json.dumps(doc)


def save_reparsed(path, new_json, attempts):
    """Write the new parse (if any) and the attempt count. Clearing markdown makes step 3 convert the document again."""
    target = DeltaTable.forName(spark, silver)
    if new_json is None:
        target.update(f"path = '{path}'", {"parse_attempts": str(attempts)})
        return
    df = spark.createDataFrame([(path, new_json)], "path string, j string").select("path", F.expr("parse_json(j)").alias("parsed"))
    (
        target.alias("t").merge(df.alias("s"), "t.path = s.path")
        .whenMatchedUpdate(set={"parsed": "s.parsed", "markdown": "NULL", "parse_attempts": str(attempts)})
        .execute()
    )

### 2b-2. Run the re-parse within the budget

Works through the plan one document at a time, the documents with the most lost pages first.

For each document: take its failed pages, cut them into calls of `reparse_chunk_pages`, parse them, pause, and stop the whole run if the rate limit is back. Pages that parsed are spliced in. A document counts as a failed pass (`parse_attempts` + 1) only if some of the pages it tried failed again, so a document that is just cut short by the budget is not penalised.

With `reparse_dry_run = yes` nothing is parsed: the plan above is all that is printed.

In [ ]:
spent, stopped = 0, False
if reparse_dry_run:
    print("Dry run: nothing re-parsed.")
elif not plan:
    print("Nothing to re-parse.")

for r in ([] if reparse_dry_run else plan):
    if spent >= reparse_page_budget or stopped:
        break
    doc_json = spark.sql(f"SELECT CAST(parsed AS STRING) FROM {silver} WHERE path = '{r.path}'").first()[0]
    failed_pages = sorted({
        er["page_id"] for er in (json.loads(doc_json).get("error_status") or [])
        if re.search(RETRYABLE, er["error_message"])
    })
    todo = failed_pages[: reparse_page_budget - spent]
    pdf_bytes = bytes(spark.table(bronze).filter(f"path = '{r.path}'").select("content").first()[0])

    fixed, still_failing = {}, set()
    for i in range(0, len(todo), reparse_chunk_pages):
        chunk = todo[i : i + reparse_chunk_pages]
        by_page, failed = reparse_pages(pdf_bytes, chunk)
        spent += len(chunk)
        fixed.update({pid: by_page[pid] for pid in chunk if pid not in failed})
        still_failing |= failed
        if len(failed) > len(chunk) / 2:
            print("More than half of the pages failed again (rate limit?). Stopping this run.")
            stopped = True
            break
        time.sleep(reparse_pause_seconds)

    attempts = r.attempts + (1 if still_failing else 0)
    save_reparsed(r.path, splice(doc_json, fixed, set(fixed)) if fixed else None, attempts)
    print(f"{r.path.split('/')[-1]}: {len(fixed)} of {len(failed_pages)} pages fixed, {len(still_failing)} failed again, attempts={attempts}")

if not reparse_dry_run:
    print(f"Pages re-parsed this run: {spent} of a budget of {reparse_page_budget}.")

## 3. Convert to Markdown with `ai_query`

One run converts every document that has no `markdown` yet. For each document: split into parts, send each part to `ai_query`, store the parts, and join them into `markdown`. The cells below define each piece, and the last one runs them in a loop.

**Challenge: documents are too big for one call.** A 10-K has 100+ pages. The serving endpoint caps the output at **10000 tokens** (`max_tokens`), so a whole filing converted in one call is silently cut off. The first test did exactly that: 150-page filings came back as one 43,000-character Markdown that stopped early.
**Solution:** split each document into **parts of `pages_per_call` pages** (default 3) and make one `ai_query` call per part. A part's input and output both stay small. The parts are joined afterwards, in order.

### 3a. The prompt

The prompt is the contract for the conversion. The model gets one part as JSON (`page_no`, `type`, `content` per element, in reading order) and must return only that part's Markdown.

| Requirement | How the prompt handles it |
| --- | --- |
| Titles and headings | `title` becomes `#`, `section_header` becomes `##` / `###` |
| Plain text | Paragraphs and Markdown lists, copied exactly |
| Tables | The parser gives tables as HTML. The prompt asks for Markdown pipe tables with a header and `\| --- \|` row, keeping every row and column |
| Metadata | Page headers, footers and page numbers are kept as italic lines |
| Page separators | Each page starts with `<!-- page N -->`, and `---` goes between pages |
| No extra text | No code fences or commentary, and nothing summarized, reworded or invented |

**Challenge: page separators are written by the model, inside a part.** Between two parts the separator is added in code (step 3d), so those are always exact. Inside a part the model has to follow the rule, so check the `<!-- page N -->` count against the page count after a run.

In [ ]:
PROMPT = """Convert these parsed elements of a PDF (JSON from ai_parse_document) into Markdown.

Each element has "page_no", "type" and "content", in reading order.

Rules:
- Keep ALL content exactly as written. Do not summarize, reword or invent anything.
- Titles become "# Title"; section headers become "##" / "###".
- Keep plain text as paragraphs and lists as Markdown lists.
- Convert tables (given as HTML) to Markdown tables (header row, | --- | separator row), keeping every row and column.
- Keep metadata such as page headers, footers and page numbers as italic lines.
- Start each page with the line "<!-- page N -->" (N = page_no). Between two pages put a blank line, "---", and a blank line.
- Output ONLY the Markdown, with no code fences or commentary.

Elements:
"""

### 3b. Split documents into parts

`build_parts(docs, size)` takes the documents to convert and returns one row per part of `size` pages with its `ai_query` prompt and the list of page numbers in the part (used later to check the markers).

1. **Explode** `parsed:document:elements` into one row per element, keeping its reading order (`pos`), `type` and `content`.
2. **Number the pages** 1, 2, 3 per document with `dense_rank` over the page id.
3. **Group pages** into parts: `part = floor((page_no - 1) / pages_per_call)`, so pages 1-3 are part 0, pages 4-6 are part 1, and so on. A page is never cut in half. `num_parts` is the number of parts of the document.
4. **Build the input**: the part's elements as JSON (`page_no`, `type`, `content`), added to the prompt.

**Challenge: the page number is not where it looks.** `ai_parse_document` v2.0 elements have no top-level `page_id`. The page is in `bbox[0].page_id`. Reading the wrong field gave NULL for every element, so every document became one page and one part, which was the cause of the truncated output above.
**Solution:** read `coalesce(bbox[0].page_id, 0)`. Verified on real data: 1,042 elements spread over 150 pages (no element without a page).

In [ ]:
def element_rows(docs):
    """One row per parsed element: path, pos (reading order), page_id, type, content, page_no (pages with content, 1, 2, 3 per document)."""
    docs.createOrReplaceTempView("docs_to_split")
    return spark.sql("""
    SELECT p.path, e.pos,
           coalesce(e.value:bbox[0].page_id::INT, 0) AS page_id,
           e.value:type::STRING AS type,
           e.value:content::STRING AS content
    FROM docs_to_split p, LATERAL variant_explode(p.parsed:document:elements) e
    """).withColumn("page_no", F.dense_rank().over(Window.partitionBy("path").orderBy("page_id")))


def expected_pages(docs, size):
    """The page numbers that belong in each part of `size` pages: path, part, pages."""
    return (
        element_rows(docs)
        .withColumn("part", F.floor((F.col("page_no") - 1) / size).cast("int"))
        .groupBy("path", "part")
        .agg(F.array_sort(F.collect_set("page_no")).alias("pages"))
    )


def build_parts(docs, size):
    """One row per part of `size` pages: path, part, num_parts, pages (the page numbers in it), prompt."""
    return (
        element_rows(docs)
        .withColumn("part", F.floor((F.col("page_no") - 1) / size).cast("int"))
        .groupBy("path", "part")
        .agg(
            F.max("page_no").alias("last_page"),
            F.array_sort(F.collect_set("page_no")).alias("pages"),
            F.array_sort(F.collect_list(F.struct("pos", "page_no", "type", "content"))).alias("els"),
        )
        # num_parts always counts parts of pages_per_call pages, also when redoing a part page by page
        .withColumn("num_parts", F.ceil(F.max("last_page").over(Window.partitionBy("path")) / pages_per_call).cast("int"))
        .withColumn("input", F.to_json(F.transform("els", lambda e: F.struct(e["page_no"], e["type"], e["content"]))))
        .withColumn("prompt", F.concat(F.lit(PROMPT), F.col("input")))
        .select("path", "part", "num_parts", "pages", "prompt")
    )

### 3b2. Drop stored parts that no longer match the parse

A stored part is only valid if its page markers are exactly the pages the **current** parse puts in that part. Two things make a stored part stale:

- **A page was re-parsed (step 2b).** New content shifts the page numbers of everything after it, so the parts from the first changed page onward have different pages than their stored markers. Parts before it are unchanged.
- **A part has wrong markers from an earlier run** (a repeated or skipped marker).

`drop_stale_parts` compares, for every stored part, the markers found in its text with the expected pages, **without any `ai_query` call**. Parts that differ are removed from `parts` and the document's `markdown` is cleared, so step 3 converts only those parts again.

This is what keeps the reconversion small after a re-parse: only the parts that really changed are paid for again.

In [ ]:
def drop_stale_parts():
    """Remove stored parts whose markers do not match the current parse; clear markdown of their documents. Returns the number removed."""
    docs = spark.table(silver).filter(f"parts IS NOT NULL AND {HAS_ELEMENTS}")
    expected = expected_pages(docs, pages_per_call)
    stale = (
        docs.select("path", F.explode("parts").alias("part", "text"))
        .join(expected, ["path", "part"], "left")
        .withColumn("found", F.expr("transform(regexp_extract_all(text, '<!-- page ([0-9]+) -->', 1), x -> cast(x AS INT))"))
        .filter("pages IS NULL OR found <> pages")
        .groupBy("path").agg(F.collect_list("part").alias("bad"))
    )
    rows = stale.collect()  # small: a document path and a few part numbers (no cache: not allowed on serverless here)
    removed = sum(len(r.bad) for r in rows)
    if removed:
        bad = spark.createDataFrame([(r.path, [int(k) for k in r.bad]) for r in rows], "path string, bad array<int>")
        (
            DeltaTable.forName(spark, silver).alias("t")
            .merge(bad.alias("s"), "t.path = s.path")
            .whenMatchedUpdate(set={
                "parts": "map_filter(t.parts, (k, v) -> NOT array_contains(s.bad, k))",
                "markdown": "NULL",
            })
            .execute()
        )
    return removed


### 3c. Call `ai_query` for the missing parts

`call_ai_query` skips parts already stored in `parts` and sends the rest to the endpoint.

- `temperature` 0 and `max_tokens` 10000 (the endpoint's cap).
- `repartition(32)` spreads the calls over the workers so they run in parallel. Without it, a small number of parts can land in a few partitions and run one after another.
- `failOnError => false` makes a bad call return an error instead of failing the whole statement.

**Challenge: a failure should not throw away finished work, and a rerun should not repeat it.**
**Solution:** only successful parts are kept. Parts that failed are not stored, so the next run (or the next batch) sends only the missing ones. `ai_query` with `failOnError => false` returns a struct with the fields `result` and `errorMessage`. The first version read a field called `response`, which does not exist, and failed with `FIELD_NOT_FOUND`.

**Challenge: choosing the model.** `databricks-claude-sonnet-4` does not exist in this workspace (`AI_FUNCTION_COMPILATION_ERROR`), and `max_tokens` above 10000 is rejected (`BAD_REQUEST`) by the Qwen endpoint.
**Solution:** list the available endpoints with `databricks serving-endpoints list`, use one of them in the `endpoint` widget, and keep `max_tokens` within its cap.

In [ ]:
def call_ai_query(parts, stored):
    """Run ai_query on parts not yet stored. `stored` has columns path and done (MAP of parts already saved)."""
    todo = (
        parts.join(stored, "path")
        .filter("done IS NULL OR NOT map_contains_key(done, part)")
    )
    return todo.repartition(32).withColumn("r", F.expr(
        f"ai_query('{endpoint}', prompt, "
        "modelParameters => named_struct('temperature', 0.0, 'max_tokens', 10000), "
        "failOnError => false)"
    )).cache()


def with_validation(called):
    """Mark each result `ok` when the call succeeded and its page markers are exactly the part's pages, once each, in order."""
    return (
        called
        .withColumn("found", F.expr("transform(regexp_extract_all(r.result, '<!-- page ([0-9]+) -->', 1), x -> cast(x AS INT))"))
        .withColumn("ok", F.expr("r.errorMessage IS NULL AND r.result IS NOT NULL AND found = pages"))
    )

### 3d. Check the page markers, redo bad parts page by page

The model writes the `<!-- page N -->` markers, and it sometimes repeats one or skips one (in the first full run, 151 of 359 documents had a different marker count than pages with content). So every result is **validated**: the markers found in the Markdown must be exactly the part's page numbers, once each, in order.

- **Valid parts** are kept as they are.
- **Invalid parts** are redone **one page per call** (`build_parts(docs, 1)`). For a single page the marker is known, so it is **added in code**, not trusted to the model. Any `<!-- page N -->` marker or stray `---` line the model wrote is removed first. This matters because the model sometimes mistakes the page number printed on the page (for example "17" on page 16) for a marker and adds one, which made 10 parts fail even page by page in the first repair run. The pages are then stitched into the part in code, in page order, with `---` between them.
- A part is only left out when its `ai_query` calls themselves fail (an error or no result). It is then retried on the next batch or run, and counted as failed.

`save_parts` then merges the accepted parts into `silver_pdfs.parts` (details in 3e).

In [ ]:
def redo_by_page(docs, bad):
    """Redo the bad parts one page per call. Returns accepted parts: path, part, num_parts, result.

    For a single page the marker is known, so it is added here and not trusted to the model: any
    `<!-- page N -->` marker or stray `---` line the model wrote is removed first (for example when it
    mistakes the printed page number for a page marker)."""
    pages = (
        build_parts(docs.join(bad.select("path").distinct(), "path"), 1)
        .withColumn("orig_part", F.floor(F.col("part") / pages_per_call).cast("int"))
        .join(bad.select("path", F.col("part").alias("orig_part")), ["path", "orig_part"])  # only pages of bad parts
    )
    no_stored = docs.select("path", F.lit(None).cast("map<int,string>").alias("done"))
    redone = (
        call_ai_query(pages, no_stored)
        .withColumn("ok", F.expr("r.errorMessage IS NULL AND r.result IS NOT NULL"))
        .withColumn("page_md", F.expr(
            "concat('<!-- page ', cast(pages[0] AS STRING), ' -->\\n', "
            "trim(regexp_replace(regexp_replace(regexp_replace(r.result, '<!-- page [0-9]+ -->', ''), '(?m)^---[ ]*$', ''), '\\n{3,}', '\\n\\n')))"
        ))
    )
    return (
        redone.groupBy("path", "orig_part", "num_parts")
        .agg(
            F.count("*").alias("n"),
            F.sum(F.col("ok").cast("int")).alias("n_ok"),
            F.array_sort(F.collect_list(F.struct("part", "page_md"))).alias("p"),
        )
        .filter("n = n_ok")  # every page of the part came back
        .select(
            "path", F.col("orig_part").alias("part"), "num_parts",
            F.concat_ws("\n\n---\n\n", F.transform("p", lambda x: x["page_md"])).alias("result"),
        )
    )


def save_parts(accepted):
    """Merge accepted parts (path, part, num_parts, result) into silver_pdfs."""
    new_parts = (
        accepted.groupBy("path", "num_parts")
        .agg(F.map_from_entries(F.collect_list(F.struct("part", "result"))).alias("new_parts"))
    )

    merged = "map_concat(coalesce(t.parts, cast(map() AS MAP<INT, STRING>)), s.new_parts)"
    (
        DeltaTable.forName(spark, silver).alias("t")
        .merge(new_parts.alias("s"), "t.path = s.path")
        .whenMatchedUpdate(set={
            "parts": merged,
            "num_parts": "s.num_parts",
            "markdown": f"CASE WHEN size({merged}) = s.num_parts THEN "
                        f"concat_ws('\\n\\n---\\n\\n', transform(array_sort(map_entries({merged})), x -> x.value)) END",
        })
        .execute()
    )

### 3e. Save the parts, build the Markdown, run all documents in batches

**Saving.** The merge adds the accepted parts to the stored map with `map_concat`. It only **updates** existing rows: step 2 already inserted every row, and the source is built from rows of the same table, so there is never anything to insert. When the map has `num_parts` entries, `markdown` is the parts ordered by part number and joined with a blank line, `---` and a blank line (added in code, so exact). A document with a missing part keeps `markdown` NULL, so only its missing parts are converted next time.

The loop converts every document with no `markdown`, `docs_per_batch` documents at a time (default 50). One job run therefore converts everything, and Spark runs the calls of each batch in parallel.

**Challenge: converting everything in one statement is risky.** Parts are only stored when the merge finishes. If one run covered all ~365 documents and failed late, every part converted in that run would be lost, and rerunning it would pay for all of those `ai_query` calls again.
**Solution:** work in batches and save each batch (3d) before starting the next. A failure loses at most one batch, and a rerun only redoes the missing parts.

**Challenge: some PDFs cannot be parsed at all.** `ai_parse_document` rejects PDFs over 500 pages (for example the PepsiCo 10-Ks) and sometimes fails a conversion. Such a document has no elements, so it never produces a part. In the first full run these documents were picked first in every batch, the batch stored nothing, and the loop stopped with 189 convertible documents still waiting.
**Solution:** step 3 only picks documents that have parsed elements. The skipped documents are listed with the parser's reason at the end of the cell.

**Challenge: a part that can never succeed would loop forever and burn money.**
**Solution:** the loop stops if a batch stores no new part. The log line shows how many parts failed. Lower `pages_per_call` if parts keep failing, for example when a table-heavy part is too large.

`docs_per_batch` is the save point, not a limit. `max_parts_per_run` (default 1000) is the budget: the loop stops after that many new parts, and the next run continues. The loop keeps going until no document is left.

In [ ]:
def convert_batch():
    """Convert up to docs_per_batch documents. Returns the number of parts that could not be stored."""
    # Cached so every step of the batch sees the same documents (limit without order is not stable).
    docs = spark.table(silver).filter(CONVERTIBLE).limit(docs_per_batch).cache()
    docs.count()
    stored = docs.select("path", F.col("parts").alias("done"))

    # First pass: parts of pages_per_call pages. Keep the ones whose page markers are right.
    first = with_validation(call_ai_query(build_parts(docs, pages_per_call), stored))
    good = first.filter("ok").select("path", "part", "num_parts", F.col("r.result").alias("result"))
    bad = first.filter("NOT ok").select("path", "part", "num_parts", "pages").cache()
    n_bad = bad.count()

    # Second pass: redo the bad parts one page per call.
    redone = redo_by_page(docs, bad).cache()
    n_redone = redone.count()

    save_parts(good.unionByName(redone))
    first.unpersist(); bad.unpersist(); redone.unpersist(); docs.unpersist()
    print(f"  {n_bad} parts had wrong or missing page markers or failed; {n_redone} fixed by redoing them page by page")
    return n_bad - n_redone


def stored_parts():
    return spark.table(silver).selectExpr("coalesce(sum(size(parts)), 0)").first()[0]

def docs_left():
    return spark.table(silver).filter(CONVERTIBLE).count()


removed = drop_stale_parts()
print(f"{removed} stored parts no longer matched the parse and were dropped (they will be converted again).")

batch, start_parts = 0, stored_parts()
while docs_left() > 0:
    batch += 1
    before = stored_parts()
    failed = convert_batch()
    print(f"batch {batch}: {stored_parts() - before} parts stored, {failed} failed, {docs_left()} documents left")
    if stored_parts() == before:
        print("No progress in this batch, stopping. Check the failed parts, or lower pages_per_call.")
        break
    if stored_parts() - start_parts >= max_parts_per_run:
        print(f"Reached max_parts_per_run ({max_parts_per_run}). Stopping. Run the job again to continue.")
        break


# PDFs that ai_parse_document could not parse (for example more than 500 pages) have no elements.
unparsed = spark.table(silver).filter("markdown IS NULL").filter(f"NOT ({HAS_ELEMENTS})")
print(f"{unparsed.count()} documents could not be parsed and were skipped:")
unparsed.selectExpr("path", "CAST(parsed:error_status[0].error_message AS STRING) AS reason").show(truncate=False)

## 4. Check the result

Prints how many documents are converted and the start of one document, so titles, headings and the first page marker can be checked by eye. For a deeper check, compare `<!-- page N -->` markers with the number of pages that had parsed content, and look at the end of a document (a filing should end with its signature block, not mid-sentence).

In [ ]:
print(spark.table(silver).filter("markdown IS NOT NULL").count(), "converted,",
      spark.table(silver).filter("markdown IS NULL").count(), "left")
r = spark.table(silver).filter("markdown IS NOT NULL").limit(1).collect()
if r:
    print(r[0]["path"]); print(r[0]["markdown"][:2000])

## Notes

- **Rerun:** safe. Step 2 skips parsed PDFs (checkpoint), step 3 skips stored parts. A redeploy does not touch the table or the checkpoint.
- **Full rebuild:** drop `silver_pdfs`, delete the checkpoint folder (`/Volumes/<catalog>/<schema>/docs/_checkpoints/silver_pdfs`) and rerun.
- **Re-run only the prompt:** `UPDATE silver_pdfs SET markdown = NULL, parts = NULL, num_parts = NULL`, then rerun. The parse is kept.
- **Not handled yet:** a PDF re-uploaded with changed content (bronze tracks files by path), and a deleted PDF (silver keeps its row). See [happy_unhappy_path.md](../doc/medallion/silver/happy_unhappy_path.md).
- **Cost:** `ai_parse_document` and `ai_query` are billed per use. A 150-page filing is about 50 `ai_query` calls. Test on a few PDFs first.